In [8]:
#Manu Sri Mogalapalli - 1302250186

# Loan Approval: Traditional Programming vs Machine Learning
**Input:** `loan_approval_input.xlsx` (Applicants + Rules sheets) → **Output:** `loan_approval_output.xlsx`

| | Traditional programming | Machine learning |
|---|---|---|
| You give the computer | **Data + Rules** | **Data + Answers** (past decisions) |
| You get back | Answers | **Rules** (a learned model) |

Nothing is typed into this code: every number comes from the input Excel file. Edit the Excel, re-run all cells, and the output Excel is rebuilt with the new numbers.

## 1. Setup (works on Google Colab and Jupyter)

In [2]:
import os, sys
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict, LeaveOneOut

INPUT_FILE  = "loan_approval_input.xlsx"
OUTPUT_FILE = "loan_approval_output.xlsx"

try:                                   # are we on Google Colab?
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists(INPUT_FILE):
    print("Upload loan_approval_input.xlsx (it is not in this Colab session yet)")
    uploaded = files.upload()          # pick the input Excel file
    INPUT_FILE = list(uploaded.keys())[0]
print("Using input file:", INPUT_FILE)

Upload loan_approval_input.xlsx (it is not in this Colab session yet)


Saving loan_approval_input.xlsx to loan_approval_input.xlsx
Using input file: loan_approval_input.xlsx


## 2. Read the input Excel

In [3]:
applicants = pd.read_excel(INPUT_FILE, sheet_name="Applicants")
rules_df   = pd.read_excel(INPUT_FILE, sheet_name="Rules")
R = dict(zip(rules_df["Parameter"], rules_df["Value"]))     # e.g. R["Min_Credit_Score"]

required = ["Applicant_ID","Name","Age","Annual_Income","Credit_Score",
            "Loan_Amount","Existing_Debt","Employment_Years","Historical_Decision"]
missing = [c for c in required if c not in applicants.columns]
assert not missing, f"Missing columns in Applicants sheet: {missing}"
applicants = applicants.dropna(subset=["Applicant_ID"]).reset_index(drop=True)

# Derived ratios (computed from the input numbers, so they update automatically)
applicants["Loan_to_Income"] = applicants["Loan_Amount"]   / applicants["Annual_Income"]
applicants["Debt_to_Income"] = applicants["Existing_Debt"] / applicants["Annual_Income"]

print(f"{len(applicants)} applicants loaded.  Rules:", R)
applicants.head()

20 applicants loaded.  Rules: {'Min_Age': 21.0, 'Min_Annual_Income': 450000.0, 'Min_Credit_Score': 650.0, 'Min_Employment_Years': 3.0, 'Max_Loan_to_Income': 4.0, 'Max_Debt_to_Income': 0.5}


,Applicant_ID,Name,Age,Annual_Income,Credit_Score,Loan_Amount,Existing_Debt,Employment_Years,Historical_Decision,Loan_to_Income,Debt_to_Income
0,1,Ananya Iyer,24,1060000,740,300000,50000,5,Approved,0.283019,0.047170
1,2,Priya Patel,33,1450000,790,800000,100000,10,Approved,0.551724,0.068966
2,3,Rohan Mehta,25,3400000,620,500000,80000,1,Rejected,0.147059,0.023529
3,4,Kavya Nair,44,1565000,790,2500000,200000,15,Approved,1.597444,0.127796
4,5,Vikram Singh,46,640000,680,1800000,150000,4,Approved,2.812500,0.234375


## 3. Traditional programming: **we write the rules**
A human decides the rules (thresholds come from the `Rules` sheet) and the program applies them to each applicant.

In [4]:
def traditional_decision(row):
    reasons = []
    if row["Age"]              <  R["Min_Age"]:               reasons.append("Age too low")
    if row["Annual_Income"]    <  R["Min_Annual_Income"]:     reasons.append("Income too low")
    if row["Credit_Score"]     <  R["Min_Credit_Score"]:      reasons.append("Credit score too low")
    if row["Employment_Years"] <  R["Min_Employment_Years"]:  reasons.append("Employment too short")
    if row["Loan_to_Income"]   >  R["Max_Loan_to_Income"]:    reasons.append("Loan too large vs income")
    if row["Debt_to_Income"]   >  R["Max_Debt_to_Income"]:    reasons.append("Existing debt too high")
    return ("Approved", "All rules passed") if not reasons else ("Rejected", "; ".join(reasons))

applicants[["Traditional_Decision","Traditional_Reason"]] = applicants.apply(
    lambda r: pd.Series(traditional_decision(r)), axis=1)
applicants[["Applicant_ID","Name","Traditional_Decision","Traditional_Reason"]]

,Applicant_ID,Name,Traditional_Decision,Traditional_Reason
0,1,Ananya Iyer,Approved,All rules passed
1,2,Priya Patel,Approved,All rules passed
2,3,Rohan Mehta,Rejected,Credit score too low; Employment too short
3,4,Kavya Nair,Approved,All rules passed
4,5,Vikram Singh,Approved,All rules passed
5,6,Ananya Rao,Rejected,Income too low; Employment too short
6,7,Arjun Reddy,Rejected,Credit score too low
7,8,Meera Nair,Approved,All rules passed
8,9,Priya Kapoor,Rejected,Income too low; Credit score too low; Employme...
9,10,Divya Kulkarni,Approved,All rules passed


## 4. Machine learning: **the computer learns the rules**
We do **not** give any thresholds. The model sees the applicants' data and the past `Historical_Decision` and learns the pattern itself (logistic regression).

In [5]:
FEATURES = ["Age","Annual_Income","Credit_Score","Loan_Amount","Existing_Debt",
            "Employment_Years","Loan_to_Income","Debt_to_Income"]
X = applicants[FEATURES]
y = (applicants["Historical_Decision"].str.strip().str.title() == "Approved").astype(int)
assert y.nunique() == 2, "Historical_Decision needs at least one Approved and one Rejected row for ML to learn."

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X, y)                                           # <- learning happens here

applicants["ML_Approval_Probability"] = model.predict_proba(X)[:, 1].round(3)
applicants["ML_Decision"] = np.where(model.predict(X) == 1, "Approved", "Rejected")

# Honest check: leave-one-out = predict each applicant using a model that never saw them
loo_pred = cross_val_predict(model, X, y, cv=LeaveOneOut())
ml_train_acc = (model.predict(X) == y).mean()
ml_loo_acc   = (loo_pred == y).mean()

# What did the model learn? (importance of each feature, larger = matters more)
coefs = pd.DataFrame({"Feature": FEATURES,
                      "Learned_Weight": model.named_steps["logisticregression"].coef_[0].round(3)})
coefs["Direction"] = np.where(coefs["Learned_Weight"] >= 0, "Pushes toward Approve", "Pushes toward Reject")
coefs = coefs.reindex(coefs["Learned_Weight"].abs().sort_values(ascending=False).index).reset_index(drop=True)
print(f"ML accuracy on the data it learned from : {ml_train_acc:.0%}")
print(f"ML accuracy on unseen applicants (LOO)  : {ml_loo_acc:.0%}")
coefs

ML accuracy on the data it learned from : 95%
ML accuracy on unseen applicants (LOO)  : 85%


,Feature,Learned_Weight,Direction
0,Credit_Score,1.833,Pushes toward Approve
1,Loan_to_Income,-0.612,Pushes toward Reject
2,Age,0.465,Pushes toward Approve
3,Debt_to_Income,-0.412,Pushes toward Reject
4,Loan_Amount,0.170,Pushes toward Approve
5,Existing_Debt,-0.146,Pushes toward Reject
6,Annual_Income,-0.054,Pushes toward Reject
7,Employment_Years,0.027,Pushes toward Approve


## 5. Compare both approaches and write the output Excel

In [6]:
applicants["Traditional_Matches_Historical"] = np.where(applicants["Traditional_Decision"] == applicants["Historical_Decision"], "Yes", "No")
applicants["ML_Matches_Historical"]          = np.where(applicants["ML_Decision"]          == applicants["Historical_Decision"], "Yes", "No")
applicants["Traditional_vs_ML_Agree"]        = np.where(applicants["Traditional_Decision"] == applicants["ML_Decision"],          "Yes", "No")

out_cols = ["Applicant_ID","Name","Age","Annual_Income","Credit_Score","Loan_Amount","Existing_Debt",
            "Employment_Years","Loan_to_Income","Debt_to_Income","Historical_Decision",
            "Traditional_Decision","Traditional_Reason","ML_Decision","ML_Approval_Probability",
            "Traditional_Matches_Historical","ML_Matches_Historical","Traditional_vs_ML_Agree"]
decisions = applicants[out_cols].copy()
decisions["Loan_to_Income"] = decisions["Loan_to_Income"].round(2)
decisions["Debt_to_Income"] = decisions["Debt_to_Income"].round(2)

n = len(decisions)
summary = pd.DataFrame({
  "Metric": ["Total applicants",
             "Approved - Historical", "Approved - Traditional program", "Approved - Machine learning",
             "Traditional matches historical (%)", "ML matches historical, in-sample (%)",
             "ML matches historical, unseen / leave-one-out (%)", "Traditional and ML agree (%)"],
  "Value":  [n,
             int((decisions["Historical_Decision"]=="Approved").sum()),
             int((decisions["Traditional_Decision"]=="Approved").sum()),
             int((decisions["ML_Decision"]=="Approved").sum()),
             round(100*(decisions["Traditional_Matches_Historical"]=="Yes").mean(),1),
             round(100*ml_train_acc,1), round(100*ml_loo_acc,1),
             round(100*(decisions["Traditional_vs_ML_Agree"]=="Yes").mean(),1)]})

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as xw:
    decisions.to_excel(xw, sheet_name="Decisions", index=False)
    summary.to_excel(xw, sheet_name="Summary", index=False)
    coefs.to_excel(xw,  sheet_name="ML_Learned_Weights", index=False)
    rules_df.to_excel(xw, sheet_name="Rules_Used", index=False)

    # ---- formatting ----
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
    head = PatternFill("solid", fgColor="1F4E78"); green = PatternFill("solid", fgColor="C6EFCE"); red = PatternFill("solid", fgColor="FFC7CE")
    for ws in xw.book.worksheets:
        for c in ws[1]:
            c.font = Font(bold=True, color="FFFFFF"); c.fill = head
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        for i, col in enumerate(ws.columns, 1):
            ws.column_dimensions[get_column_letter(i)].width = min(max(len(str(c.value)) if c.value is not None else 0 for c in col) + 3, 46)
        ws.freeze_panes = "A2"
    ws = xw.book["Decisions"]
    for row in ws.iter_rows(min_row=2):
        for c in row:
            if c.value in ("Approved","Yes"): c.fill = green
            elif c.value in ("Rejected","No"): c.fill = red
            if ws.cell(1, c.column).value in ("Annual_Income","Loan_Amount","Existing_Debt"): c.number_format = "#,##0"

print("Saved:", OUTPUT_FILE)
summary

Saved: loan_approval_output.xlsx


,Metric,Value
0,Total applicants,20.0
1,Approved - Historical,8.0
2,Approved - Traditional program,8.0
3,Approved - Machine learning,7.0
4,Traditional matches historical (%),100.0
5,"ML matches historical, in-sample (%)",95.0
6,"ML matches historical, unseen / leave-one-out (%)",85.0
7,Traditional and ML agree (%),95.0


## 6. Preview the output (and download it if you are on Colab)

In [7]:
display(pd.read_excel(OUTPUT_FILE, sheet_name="Decisions"))
if IN_COLAB:
    files.download(OUTPUT_FILE)        # saves the output Excel to your computer

,Applicant_ID,Name,Age,Annual_Income,Credit_Score,Loan_Amount,Existing_Debt,Employment_Years,Loan_to_Income,Debt_to_Income,Historical_Decision,Traditional_Decision,Traditional_Reason,ML_Decision,ML_Approval_Probability,Traditional_Matches_Historical,ML_Matches_Historical,Traditional_vs_ML_Agree
0,1,Ananya Iyer,24,1060000,740,300000,50000,5,0.28,0.05,Approved,Approved,All rules passed,Approved,0.762,Yes,Yes,Yes
1,2,Priya Patel,33,1450000,790,800000,100000,10,0.55,0.07,Approved,Approved,All rules passed,Approved,0.932,Yes,Yes,Yes
2,3,Rohan Mehta,25,3400000,620,500000,80000,1,0.15,0.02,Rejected,Rejected,Credit score too low; Employment too short,Rejected,0.177,Yes,Yes,Yes
3,4,Kavya Nair,44,1565000,790,2500000,200000,15,1.60,0.13,Approved,Approved,All rules passed,Approved,0.945,Yes,Yes,Yes
4,5,Vikram Singh,46,640000,680,1800000,150000,4,2.81,0.23,Approved,Approved,All rules passed,Rejected,0.432,Yes,No,No
5,6,Ananya Rao,22,430000,670,1500000,100000,2,3.49,0.23,Rejected,Rejected,Income too low; Employment too short,Rejected,0.119,Yes,Yes,Yes
6,7,Arjun Reddy,40,2100000,540,700000,300000,12,0.33,0.14,Rejected,Rejected,Credit score too low,Rejected,0.050,Yes,Yes,Yes
7,8,Meera Nair,31,780000,750,2000000,90000,7,2.56,0.12,Approved,Approved,All rules passed,Approved,0.728,Yes,Yes,Yes
8,9,Priya Kapoor,28,286100,630,400000,20000,1,1.40,0.07,Rejected,Rejected,Income too low; Credit score too low; Employme...,Rejected,0.187,Yes,Yes,Yes
9,10,Divya Kulkarni,36,1080000,780,3000000,400000,9,2.78,0.37,Approved,Approved,All rules passed,Approved,0.738,Yes,Yes,Yes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>